In [0]:
from pyspark.sql.functions import lit
catalog_name = "multiplex"

bronze_schema = "multiplex_1_bronze"
silver_schema = "multiplex_2_silver"
gold_schema = "multiplex_3_gold"

volume_name = "business_events"

source_path = "/Volumes/databricks_simulated_retail_customer_data/v02/business_daily_events"
sink_path = f"/Volumes/{catalog_name}/{bronze_schema}"

date_on_file = "2025-11-03"
file_name = f"retail_business_events_{date_on_file}.json"

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{gold_schema}")

spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{bronze_schema}.{volume_name}")

In [0]:

dbutils.fs.cp(f"{source_path}/{file_name}", f"{sink_path}/{volume_name}/{file_name}")


In [0]:
## Check file
sql_result = (
    spark.sql(f"LIST '{sink_path}/{volume_name}/'")
    .withColumn('volume', lit(f"{volume_name}"))
)

display(sql_result)

In [0]:
sql_query_result = (
    spark.sql(f"""
        SELECT '{volume_name}' as volume_name,
            COUNT(*) as total_rows,
            _metadata.file_name as file_name
        FROM read_files('{sink_path}/{volume_name}/')
        GROUP BY _metadata.file_name
    """)
)

display(sql_query_result)

In [0]:
spark.sql(f"""
    DESCRIBE EXTENDED multiplex.multiplex_3_gold.logistics_delta_sink
""").display()

In [0]:
spark.sql("""
        SELECT source_file,
                COUNT(*) total_rows
        FROM multiplex.multiplex_1_bronze.bronze_demo
        GROUP BY source_file
        ORDER BY 1
""").display()

In [0]:
spark.sql("""
        SELECT source_file, event_group,                
                COUNT(*) total_rows
        FROM multiplex.multiplex_1_bronze.bronze_demo
        GROUP BY source_file, event_group
        ORDER BY 1,2
""").display()